# Smart Supermarket Product Identification — Kaggle runner
Runs the project's scripts on a Kaggle GPU. All code lives in the GitHub repository; this notebook only
clones it and calls the scripts in order.

**Before running**
1. *Add Input* → **`diyer22/retail-product-checkout-dataset`**
2. *Settings* → **Accelerator: GPU T4 x2** (or P100), **Internet: On**
3. Put your repository URL below, then *Save Version → Save & Run All (Commit)*.

**Session plan** (every step skips work that is already finished):

| Session | Settings | Runs |
|---|---|---|
| 1 | `run_cv=false` | prepare → train → validate → test → outcomes → export |
| 2 | `resume_from=<session 1 output>` `cv_folds_to_run=[0,1,2]` | cross-validation folds 0–2 → outcomes |
| 3 | `resume_from=<session 2 output>` `cv_folds_to_run=[3,4]` | folds 3–4 → final outcomes + export |

To continue from an earlier session: *Add Input → Your Work → (this notebook) → Output*, then use its path
(e.g. `/kaggle/input/smart-checkout-runner`) as `resume_from`.

In [ ]:
# ---------------- settings ----------------
REPO_URL = "https://github.com/Dhanushka0626/Supermarket-product-identification-.git"
BRANCH = "main"
SETTINGS = [
    "run_cv=false",                  # session 1; later sessions: "run_cv=true"
    # "resume_from=/kaggle/input/smart-checkout-runner",
    # "cv_folds_to_run=[0,1,2]",
    # "epochs=10", "test_limit=2000",  # quick trial run
]
# -------------------------------------------
SET = " ".join(SETTINGS)
print("Settings:", SET or "(defaults from configs/config.yaml)")

In [ ]:
!rm -rf /tmp/project && git clone -q --depth 1 -b {BRANCH} {REPO_URL} /tmp/project
# Private repository? Upload the repo as a Kaggle dataset instead and use:
# !cp -r /kaggle/input/<your-repo-dataset> /tmp/project
%cd /tmp/project
!pip install -q ultralytics
!git log --oneline -3

## Phase 1 — data preparation (Member 1)

In [ ]:
!python scripts/prepare_data.py --set {SET}

## Phase 2 — training (Member 1)

In [ ]:
!python scripts/train.py --set {SET}

## Phase 3 — hold-out validation + threshold tuning (Member 2)

In [ ]:
!python scripts/validate.py --set {SET}

## Phase 4 — final test on test2019 (Member 2)

In [ ]:
!python scripts/evaluate_test.py --set {SET}

## Phase 5 — K-fold cross-validation (Member 2)

In [ ]:
!python scripts/cross_validate.py --set {SET}

## Phase 6 — outcomes package + export for local use (Member 2)

In [ ]:
!python scripts/build_outcomes.py --set {SET} > /tmp/outcomes.log && tail -n 5 /tmp/outcomes.log
!python scripts/export_model.py --set {SET}

## Results

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

OUT = Path("/kaggle/working/outcomes")
display(Markdown((OUT / "outcomes_report.md").read_text()))
for name in ["training_curves.png", "conf_threshold.png", "cm_test.png", "cv_summary.png",
             "cv_per_class_ap50.png", "example_hard.jpg", "example_hard_chart.png"]:
    f = OUT / "figures" / name
    if f.exists():
        print(name); display(Image(filename=str(f), width=900))